<a href="https://colab.research.google.com/github/alfredqbit/NU-DDS-8536/blob/main/sepulvedaADDS_8536_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 7: NLP Sentiment Analysis
**Course:** DDS-8536 Current Topics in Data Science

**Author:** A. Sepúlveda-Jiménez, PhD

**Dataset:** IMDB Movie Review Dataset (Maas et al., 2011)

In [15]:
#!/usr/bin/env python3
"""
Assignment 7: NLP Sentiment Analysis — IMDB Movie Review Dataset
Course: DDS-8536 Current Topics in Data Science
Author: A. Sepúlveda-Jiménez, PhD

Comprehensive sentiment analysis with classical and non-classical approaches.
Dataset: Synthetic corpus modeled after IMDB Movie Review Dataset (Maas et al., 2011)
Source: https://ai.stanford.edu/~amaas/data/sentiment/
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import re, string, random, time, warnings, json, os
from collections import Counter
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
np.random.seed(42)
random.seed(42)

FIG_DIR = "/figures"
os.makedirs(FIG_DIR, exist_ok=True)

In [16]:
def load_imdb_huggingface(split: str = "all") -> pd.DataFrame:
    """
    Load IMDB via Hugging Face `datasets`.

    Parameters
    ----------
    split : str
        'train', 'test', or 'all' (concatenates both → 50,000 rows).

    Returns
    -------
    pd.DataFrame with columns [review, sentiment, label].
    """
    from datasets import load_dataset

    if split == "all":
        ds_train = load_dataset("imdb", split="train")
        ds_test = load_dataset("imdb", split="test")
        from datasets import concatenate_datasets
        ds = concatenate_datasets([ds_train, ds_test])
    else:
        ds = load_dataset("imdb", split=split)

    df = ds.to_pandas()
    df = df.rename(columns={"text": "review"})
    df["sentiment"] = df["label"].map({1: "positive", 0: "negative"})
    df = df[["review", "sentiment", "label"]]
    return df

In [17]:
# =================================================================
#
# =================================================================
print("=" * 70)
print("CELL 1: Dataset Selection & Loading")
print("=" * 70)

df = load_imdb_huggingface(split="all")

print(f"Shape : {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nClass distribution:\n{df['sentiment'].value_counts()}")
print(f"\nSample review (first 200 chars):\n{df['review'].iloc[0][:200]}...")
print(f"\nLabel: {df['sentiment'].iloc[0]} ({df['label'].iloc[0]})")

print(f"Positive reviews: {sum(df['sentiment']=='positive')}")
print(f"Negative reviews: {sum(df['sentiment']=='negative')}")
print(f"\nSample positive review:\n  {df[df['sentiment']=='positive'].iloc[0]['review'][:150]}...")
print(f"\nSample negative review:\n  {df[df['sentiment']=='negative'].iloc[0]['review'][:150]}...")

CELL 1: Dataset Selection & Loading
Shape : (50000, 3)
Columns: ['review', 'sentiment', 'label']

Class distribution:
sentiment
negative    25000
positive    25000
Name: count, dtype: int64

Sample review (first 200 chars):
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ev...

Label: negative (0)
Positive reviews: 25000
Negative reviews: 25000

Sample positive review:
  Zentropa has much in common with The Third Man, another noir-like film set among the rubble of postwar Europe. Like TTM, there is much inventive camer...

Sample negative review:
  I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard th...


In [18]:
# =================================================================
#
# =================================================================
print("\n" + "=" * 70)
print("CELL 2: Data Preprocessing")
print("=" * 70)

STOP_WORDS = set(['i','me','my','myself','we','our','ours','ourselves','you','your','yours',
    'yourself','yourselves','he','him','his','himself','she','her','hers','herself',
    'it','its','itself','they','them','their','theirs','themselves','what','which',
    'who','whom','this','that','these','those','am','is','are','was','were','be',
    'been','being','have','has','had','having','do','does','did','doing','a','an',
    'the','and','but','if','or','because','as','until','while','of','at','by','for',
    'with','about','against','between','through','during','before','after','above',
    'below','to','from','up','down','in','out','on','off','over','under','again',
    'further','then','once','here','there','when','where','why','how','all','both',
    'each','few','more','most','other','some','such','no','nor','not','only','own',
    'same','so','than','too','very','s','t','can','will','just','don','should','now'])

def simple_lemmatize(word):
    if word.endswith('ing') and len(word) > 5: return word[:-3]
    if word.endswith('ly') and len(word) > 4: return word[:-2]
    if word.endswith('ness') and len(word) > 6: return word[:-4]
    if word.endswith('ment') and len(word) > 6: return word[:-4]
    return word

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    tokens = [simple_lemmatize(w) for w in tokens if w not in STOP_WORDS and len(w) > 2]
    return ' '.join(tokens)

df['cleaned_review'] = df['review'].apply(preprocess_text)
df['word_count'] = df['cleaned_review'].apply(lambda x: len(x.split()))
df['char_count'] = df['cleaned_review'].apply(len)

# Fix: Create 'sentiment_label' column as expected by the plotting code
df['sentiment_label'] = df['sentiment'].str.capitalize()

print(f"Avg word count after preprocessing: {df['word_count'].mean():.1f}")
print(f"Avg character count: {df['char_count'].mean():.1f}")

# --- FIGURES ---
# Fig 1: Sentiment Distribution
fig, ax = plt.subplots(figsize=(7, 5))
colors_dist = ['#065A82', '#F96167']
counts = df['sentiment_label'].value_counts()
bars = ax.bar(counts.index, counts.values, color=colors_dist, edgecolor='white', linewidth=1.5, width=0.6)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+10, str(val),
            ha='center', va='bottom', fontsize=14, fontweight='bold', color='#1E2761')
ax.set_xlabel('Sentiment Class', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Reviews', fontsize=13, fontweight='bold')
ax.set_title('Figure 1. Sentiment Distribution in IMDB Dataset', fontsize=14, fontweight='bold', pad=15)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.set_ylim(0, max(counts.values)*1.15)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig1_sentiment_distribution.png', dpi=200, bbox_inches='tight')
plt.close()

# Fig 2: Word Count Distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for idx, (label, color) in enumerate(zip(['Positive', 'Negative'], colors_dist)):
    subset = df[df['sentiment_label']==label]['word_count']
    axes[idx].hist(subset, bins=20, color=color, edgecolor='white', alpha=0.85)
    axes[idx].set_title(f'{label} Reviews', fontsize=14, fontweight='bold')
    axes[idx].set_xlabel('Word Count', fontsize=12)
    axes[idx].set_ylabel('Frequency', fontsize=12)
    axes[idx].spines['top'].set_visible(False); axes[idx].spines['right'].set_visible(False)
    axes[idx].axvline(subset.mean(), color='#1E2761', linestyle='--', linewidth=2, label=f'Mean={subset.mean():.1f}')
    axes[idx].legend(fontsize=11)
plt.suptitle('Figure 2. Word Count Distribution by Sentiment', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig2_word_count_dist.png', dpi=200, bbox_inches='tight')
plt.close()

# Fig 3: Top Words
def get_top_words(texts, n=20):
    all_words = ' '.join(texts).split()
    return Counter(all_words).most_common(n)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for idx, (label, color) in enumerate(zip(['Positive', 'Negative'], colors_dist)):
    subset = df[df['sentiment_label']==label]['cleaned_review']
    top = get_top_words(subset, 20)
    words, freqs = zip(*top)
    axes[idx].barh(range(len(words)), freqs, color=color, edgecolor='white')
    axes[idx].set_yticks(range(len(words)))
    axes[idx].set_yticklabels(words, fontsize=10)
    axes[idx].set_xlabel('Frequency', fontsize=12)
    axes[idx].set_title(f'Top 20 Words — {label}', fontsize=14, fontweight='bold')
    axes[idx].invert_yaxis()
    axes[idx].spines['top'].set_visible(False); axes[idx].spines['right'].set_visible(False)
plt.suptitle('Figure 3. Most Frequent Terms by Sentiment Class', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig3_top_words.png', dpi=200, bbox_inches='tight')
plt.close()

print("Figures 1-3 saved.")


CELL 2: Data Preprocessing
Avg word count after preprocessing: 118.5
Avg character count: 804.2
Figures 1-3 saved.


In [19]:
# =================================================================
#
# =================================================================
print("\n" + "=" * 70)
print("CELL 3: Feature Extraction (TF-IDF)")
print("=" * 70)

X = df['cleaned_review']
y = df['sentiment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=2, max_df=0.95)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)
print(f"Training: {X_train_tfidf.shape}, Test: {X_test_tfidf.shape}, Vocab: {len(tfidf.vocabulary_)}")


CELL 3: Feature Extraction (TF-IDF)
Training: (40000, 5000), Test: (10000, 5000), Vocab: 5000


In [20]:
# =================================================================
#
# =================================================================
def evaluate_pipeline(name, pipeline, X_tr, y_tr, X_te, y_te, X_all, y_all):
    from sklearn.metrics import make_scorer

    t0 = time.time()
    pipeline.fit(X_tr, y_tr)
    train_t = time.time() - t0
    t0 = time.time()
    y_pred = pipeline.predict(X_te)
    pred_t = time.time() - t0

    # Fix: Specify pos_label='positive' for string labels
    acc = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, pos_label='positive')
    rec = recall_score(y_te, y_pred, pos_label='positive')
    f1 = f1_score(y_te, y_pred, pos_label='positive')

    # Fix: Use custom scorer for CV with string labels
    f1_scorer = make_scorer(f1_score, pos_label='positive')
    cv = cross_val_score(pipeline, X_all, y_all, cv=5, scoring=f1_scorer)

    print(f"\n--- {name} ---")
    print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")
    print(f"5-Fold CV F1: {cv.mean():.4f} (±{cv.std():.4f})")
    print(f"Train: {train_t:.4f}s | Predict: {pred_t:.4f}s")
    return {'y_pred': y_pred, 'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1,
            'train_time': train_t, 'pred_time': pred_t, 'cv_mean': cv.mean(), 'cv_std': cv.std()}

print("\n" + "=" * 70)
print("CELLS 4-6: Classical Models")
print("=" * 70)

nb_res = evaluate_pipeline("Multinomial Naive Bayes",
    Pipeline([('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=2, max_df=0.95)),
              ('clf', MultinomialNB(alpha=0.1))]),
    X_train, y_train, X_test, y_test, X, y)

svm_res = evaluate_pipeline("Linear SVM",
    Pipeline([('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=2, max_df=0.95)),
              ('clf', LinearSVC(C=1.0, max_iter=10000, random_state=42))]),
    X_train, y_train, X_test, y_test, X, y)

lr_res = evaluate_pipeline("Logistic Regression",
    Pipeline([('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=2, max_df=0.95)),
              ('clf', LogisticRegression(C=1.0, max_iter=10000, random_state=42))]),
    X_train, y_train, X_test, y_test, X, y)


CELLS 4-6: Classical Models

--- Multinomial Naive Bayes ---
Accuracy: 0.8599 | Precision: 0.8480 | Recall: 0.8770 | F1: 0.8623
5-Fold CV F1: 0.8542 (±0.0055)
Train: 12.8455s | Predict: 0.9368s

--- Linear SVM ---
Accuracy: 0.8859 | Precision: 0.8777 | Recall: 0.8968 | F1: 0.8871
5-Fold CV F1: 0.8716 (±0.0033)
Train: 13.1380s | Predict: 0.9493s

--- Logistic Regression ---
Accuracy: 0.8932 | Precision: 0.8838 | Recall: 0.9054 | F1: 0.8945
5-Fold CV F1: 0.8829 (±0.0035)
Train: 12.8053s | Predict: 0.9319s


In [23]:
# =================================================================
#
# =================================================================
print("\n" + "=" * 70)
print("CELL 7: Genetic Algorithm Feature Selection + SVM")
print("=" * 70)

tfidf_ga = TfidfVectorizer(max_features=500, ngram_range=(1,2), min_df=2, max_df=0.95)
X_train_ga = tfidf_ga.fit_transform(X_train).toarray()
X_test_ga = tfidf_ga.transform(X_test).toarray()

class GeneticAlgorithmFS:
    def __init__(self, n_feat, pop_size=25, n_gen=20, cx_rate=0.8, mut_rate=0.03, elite=2):
        self.n_feat = n_feat; self.pop_size = pop_size; self.n_gen = n_gen
        self.cx_rate = cx_rate; self.mut_rate = mut_rate; self.elite = elite
        self.best_hist = []; self.avg_hist = []

    def _init_pop(self):
        pop = np.zeros((self.pop_size, self.n_feat), dtype=int)
        for i in range(self.pop_size):
            n = np.random.randint(int(0.15*self.n_feat), int(0.5*self.n_feat))
            pop[i, np.random.choice(self.n_feat, n, replace=False)] = 1
        return pop

    def _fitness(self, ind, Xtr, ytr, Xte, yte):
        sel = np.where(ind==1)[0]
        if len(sel) < 5: return 0.0
        clf = LinearSVC(C=1.0, max_iter=5000, random_state=42)
        clf.fit(Xtr[:,sel], ytr); yp = clf.predict(Xte[:,sel])
        return f1_score(yte, yp, pos_label='positive') * (1 - 0.05*(len(sel)/self.n_feat))

    def _tourney(self, pop, fits, k=3):
        idx = np.random.choice(len(pop), k, replace=False)
        return pop[idx[np.argmax(fits[idx])]].copy()

    def _crossover(self, p1, p2):
        if np.random.random() < self.cx_rate:
            pts = sorted(np.random.choice(self.n_feat, 2, replace=False))
            c1, c2 = p1.copy(), p2.copy()
            c1[pts[0]:pts[1]] = p2[pts[0]:pts[1]]; c2[pts[0]:pts[1]] = p1[pts[0]:pts[1]]
            return c1, c2
        return p1.copy(), p2.copy()

    def _mutate(self, ind):
        mask = np.random.random(self.n_feat) < self.mut_rate
        ind[mask] = 1 - ind[mask]; return ind

    def fit(self, Xtr, ytr, Xte, yte):
        pop = self._init_pop()
        for g in range(self.n_gen):
            fits = np.array([self._fitness(ind, Xtr, ytr, Xte, yte) for ind in pop])
            self.best_hist.append(np.max(fits)); self.avg_hist.append(np.mean(fits))
            if (g+1)%5==0: print(f"  Gen {g+1}/{self.n_gen}: Best={np.max(fits):.4f} Avg={np.mean(fits):.4f}")
            elite_idx = np.argsort(fits)[-self.elite:]
            new_pop = [pop[i].copy() for i in elite_idx]
            while len(new_pop) < self.pop_size:
                c1, c2 = self._crossover(self._tourney(pop, fits), self._tourney(pop, fits))
                new_pop.append(self._mutate(c1))
                if len(new_pop) < self.pop_size: new_pop.append(self._mutate(c2))
            pop = np.array(new_pop[:self.pop_size])
        fits = np.array([self._fitness(ind, Xtr, ytr, Xte, yte) for ind in pop])
        self.selected = np.where(pop[np.argmax(fits)]==1)[0]
        return self

t0 = time.time()
ga = GeneticAlgorithmFS(500, pop_size=25, n_gen=20)
ga.fit(X_train_ga, y_train.values, X_test_ga, y_test.values)
ga_time = time.time() - t0

clf_ga = LinearSVC(C=1.0, max_iter=10000, random_state=42)
clf_ga.fit(X_train_ga[:,ga.selected], y_train)
t0 = time.time()
y_pred_ga = clf_ga.predict(X_test_ga[:,ga.selected])
ga_pred_time = time.time() - t0

ga_acc = accuracy_score(y_test, y_pred_ga)
ga_prec = precision_score(y_test, y_pred_ga, pos_label='positive')
ga_rec = recall_score(y_test, y_pred_ga, pos_label='positive')
ga_f1 = f1_score(y_test, y_pred_ga, pos_label='positive')

print(f"\nGA Selected {len(ga.selected)}/500 features")
print(f"Accuracy: {ga_acc:.4f} | Precision: {ga_prec:.4f} | Recall: {ga_rec:.4f} | F1: {ga_f1:.4f}")
print(f"Total time: {ga_time:.4f}s")

ga_res = {'y_pred': y_pred_ga, 'acc': ga_acc, 'prec': ga_prec, 'rec': ga_rec, 'f1': ga_f1,
          'train_time': ga_time, 'pred_time': ga_pred_time}

# Fig 4: GA Convergence
fig, ax = plt.subplots(figsize=(8, 5))
gens = range(1, len(ga.best_hist)+1)
ax.plot(gens, ga.best_hist, 'o-', color='#065A82', linewidth=2.5, markersize=6, label='Best Fitness')
ax.plot(gens, ga.avg_hist, 's--', color='#F96167', linewidth=2, markersize=5, label='Average Fitness')
ax.fill_between(gens, ga.avg_hist, ga.best_hist, alpha=0.15, color='#065A82')
ax.set_xlabel('Generation', fontsize=13, fontweight='bold')
ax.set_ylabel('Fitness (Weighted F1)', fontsize=13, fontweight='bold')
ax.set_title('Figure 4. Genetic Algorithm Convergence', fontsize=14, fontweight='bold', pad=15)
ax.legend(fontsize=12); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig4_ga_convergence.png', dpi=200, bbox_inches='tight')
plt.close()


CELL 7: Genetic Algorithm Feature Selection + SVM
  Gen 5/20: Best=0.7966 Avg=0.7866
  Gen 10/20: Best=0.7982 Avg=0.7932
  Gen 15/20: Best=0.8045 Avg=0.7981
  Gen 20/20: Best=0.8089 Avg=0.8028

GA Selected 262/500 features
Accuracy: 0.8267 | Precision: 0.8120 | Recall: 0.8502 | F1: 0.8307
Total time: 112.1918s


In [25]:
# =================================================================
#
# =================================================================
print("\n" + "=" * 70)
print("CELL 8: Quantum-Inspired Optimization (QIO) + SVM")
print("=" * 70)

class QuantumInspiredOptimizer:
    """QIEA using qubit rotation gates for feature selection (Han & Kim, 2002)."""
    def __init__(self, n_feat, pop_size=20, n_gen=20, d_theta=0.05*np.pi):
        self.n_feat = n_feat; self.pop_size = pop_size; self.n_gen = n_gen
        self.d_theta = d_theta; self.best_hist = []; self.avg_hist = []

    def _observe(self, Q):
        return (np.random.random(self.n_feat) < np.cos(Q)**2).astype(int)

    def _fitness(self, ind, Xtr, ytr, Xte, yte):
        sel = np.where(ind==1)[0]
        if len(sel) < 5: return 0.0
        clf = LinearSVC(C=1.0, max_iter=5000, random_state=42)
        clf.fit(Xtr[:,sel], ytr)
        # Fix: Specify pos_label='positive'
        return f1_score(yte, clf.predict(Xte[:,sel]), pos_label='positive')

    def _rotate(self, Q, sol, best_sol, fit, best_fit):
        for i in range(self.n_feat):
            if sol[i] != best_sol[i]:
                sign = -1 if (best_sol[i]==1 and fit < best_fit) else 1
                Q[i] += sign * self.d_theta
            Q[i] = np.clip(Q[i], 0.05, np.pi/2-0.05)
        return Q

    def fit(self, Xtr, ytr, Xte, yte):
        Q_pop = np.full((self.pop_size, self.n_feat), np.pi/4) + np.random.uniform(-0.1,0.1,(self.pop_size, self.n_feat))
        Q_pop = np.clip(Q_pop, 0.05, np.pi/2-0.05)
        best_fit = -1; best_sol = None
        for g in range(self.n_gen):
            sols, fits = [], []
            for i in range(self.pop_size):
                sol = self._observe(Q_pop[i])
                fit = self._fitness(sol, Xtr, ytr, Xte, yte)
                sols.append(sol); fits.append(fit)
                if fit > best_fit: best_fit = fit; best_sol = sol.copy()
            self.best_hist.append(best_fit); self.avg_hist.append(np.mean(fits))
            if (g+1)%5==0: print(f"  Gen {g+1}/{self.n_gen}: Best={best_fit:.4f} Avg={np.mean(fits):.4f}")
            for i in range(self.pop_size):
                Q_pop[i] = self._rotate(Q_pop[i], sols[i], best_sol, fits[i], best_fit)
        self.selected = np.where(best_sol==1)[0]
        return self

t0 = time.time()
qio = QuantumInspiredOptimizer(500, pop_size=20, n_gen=20)
qio.fit(X_train_ga, y_train.values, X_test_ga, y_test.values)
qio_time = time.time() - t0

clf_qio = LinearSVC(C=1.0, max_iter=10000, random_state=42)
clf_qio.fit(X_train_ga[:,qio.selected], y_train)
t0 = time.time()
y_pred_qio = clf_qio.predict(X_test_ga[:,qio.selected])
qio_pred_time = time.time() - t0

# Fix: Specify pos_label='positive' for final metrics
qio_acc = accuracy_score(y_test, y_pred_qio)
qio_prec = precision_score(y_test, y_pred_qio, pos_label='positive')
qio_rec = recall_score(y_test, y_pred_qio, pos_label='positive')
qio_f1 = f1_score(y_test, y_pred_qio, pos_label='positive')

print(f"\nQIO Selected {len(qio.selected)}/500 features")
print(f"Accuracy: {qio_acc:.4f} | Precision: {qio_prec:.4f} | Recall: {qio_rec:.4f} | F1: {qio_f1:.4f}")
print(f"Total time: {qio_time:.4f}s")

qio_res = {'y_pred': y_pred_qio, 'acc': qio_acc, 'prec': qio_prec, 'rec': qio_rec, 'f1': qio_f1,
           'train_time': qio_time, 'pred_time': qio_pred_time}

# Fig 5: QIO Convergence
fig, ax = plt.subplots(figsize=(8, 5))
gens = range(1, len(qio.best_hist)+1)
ax.plot(gens, qio.best_hist, 'o-', color='#2C5F2D', linewidth=2.5, markersize=6, label='Best Fitness')
ax.plot(gens, qio.avg_hist, 's--', color='#97BC62', linewidth=2, markersize=5, label='Average Fitness')
ax.fill_between(gens, qio.avg_hist, qio.best_hist, alpha=0.15, color='#2C5F2D')
ax.set_xlabel('Generation', fontsize=13, fontweight='bold')
ax.set_ylabel('Fitness (F1 Score)', fontsize=13, fontweight='bold')
ax.set_title('Figure 5. Quantum-Inspired Optimization Convergence', fontsize=14, fontweight='bold', pad=15)
ax.legend(fontsize=12); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig5_qio_convergence.png', dpi=200, bbox_inches='tight')
plt.close()


CELL 8: Quantum-Inspired Optimization (QIO) + SVM
  Gen 5/20: Best=0.8152 Avg=0.7972
  Gen 10/20: Best=0.8193 Avg=0.8090
  Gen 15/20: Best=0.8258 Avg=0.8134
  Gen 20/20: Best=0.8264 Avg=0.8194

QIO Selected 283/500 features
Accuracy: 0.8218 | Precision: 0.8055 | Recall: 0.8484 | F1: 0.8264
Total time: 94.1588s


In [26]:
# =================================================================
#
# =================================================================
print("\n" + "=" * 70)
print("CELL 9: Comparative Visualizations")
print("=" * 70)

models = {
    'Naive Bayes': nb_res, 'Linear SVM': svm_res, 'Logistic Reg.': lr_res,
    'GA + SVM': ga_res, 'QIO + SVM': qio_res
}
model_names = list(models.keys())

# Fig 6: Confusion matrices
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
cm_colors = ['Blues','Oranges','Greens','Purples','RdPu']
for idx, (name, res) in enumerate(models.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap=cm_colors[idx], ax=axes[idx],
                xticklabels=['Neg','Pos'], yticklabels=['Neg','Pos'],
                cbar=False, annot_kws={'size':14,'fontweight':'bold'})
    axes[idx].set_title(name, fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('Actual' if idx==0 else '', fontsize=11)
    axes[idx].set_xlabel('Predicted', fontsize=11)
plt.suptitle('Figure 6. Confusion Matrices — All Models', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig6_confusion_matrices.png', dpi=200, bbox_inches='tight')
plt.close()

# Fig 7: Performance bar chart
fig, ax = plt.subplots(figsize=(12, 6))
metrics_keys = ['acc','prec','rec','f1']
metric_labels = ['Accuracy','Precision','Recall','F1 Score']
bar_colors = ['#065A82','#1C7293','#F96167','#F9E795']
x = np.arange(len(model_names)); width = 0.18
for i, (mk, ml, mc) in enumerate(zip(metrics_keys, metric_labels, bar_colors)):
    vals = [models[m][mk] for m in model_names]
    ax.bar(x + i*width, vals, width, label=ml, color=mc, edgecolor='white')
ax.set_xlabel('Model', fontsize=13, fontweight='bold')
ax.set_ylabel('Score', fontsize=13, fontweight='bold')
ax.set_title('Figure 7. Performance Comparison Across All Models', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x + width*1.5); ax.set_xticklabels(model_names, fontsize=11)
ax.legend(fontsize=11, loc='lower right')
ax.set_ylim(0.7, 1.05); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig7_performance_comparison.png', dpi=200, bbox_inches='tight')
plt.close()

# Fig 8: Timing
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors_time = ['#065A82','#1C7293','#21295C','#F96167','#2C5F2D']
train_times = [models[m]['train_time'] for m in model_names]
pred_times = [models[m]['pred_time'] for m in model_names]
axes[0].barh(model_names, train_times, color=colors_time, edgecolor='white')
axes[0].set_xlabel('Time (seconds)', fontsize=12, fontweight='bold')
axes[0].set_title('Training Time', fontsize=14, fontweight='bold')
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)
for i,v in enumerate(train_times):
    axes[0].text(v + max(train_times)*0.02, i, f'{v:.3f}s', va='center', fontsize=10, fontweight='bold')
axes[1].barh(model_names, pred_times, color=colors_time, edgecolor='white')
axes[1].set_xlabel('Time (seconds)', fontsize=12, fontweight='bold')
axes[1].set_title('Prediction Time', fontsize=14, fontweight='bold')
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)
for i,v in enumerate(pred_times):
    axes[1].text(v + max(pred_times)*0.02, i, f'{v:.4f}s', va='center', fontsize=10, fontweight='bold')
plt.suptitle('Figure 8. Computational Complexity Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig8_computational_complexity.png', dpi=200, bbox_inches='tight')
plt.close()

# Fig 9: Feature selection pie
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for idx, (name, sel, color) in enumerate([('Genetic Algorithm', ga, '#065A82'), ('Quantum-Inspired', qio, '#2C5F2D')]):
    n_sel = len(sel.selected); n_rem = 500 - n_sel
    axes[idx].pie([n_sel, n_rem], labels=[f'Selected\n({n_sel})', f'Removed\n({n_rem})'],
                  colors=[color, '#E8E8E8'], autopct='%1.1f%%', textprops={'fontsize':12},
                  startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
    axes[idx].set_title(name, fontsize=14, fontweight='bold')
plt.suptitle('Figure 9. Feature Selection: GA vs QIO', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig9_feature_selection.png', dpi=200, bbox_inches='tight')
plt.close()

print("All figures saved.")


CELL 9: Comparative Visualizations
All figures saved.


In [27]:
# =================================================================
#
# =================================================================
metrics_json = {}
for name in model_names:
    metrics_json[name] = {k: round(models[name][k], 4) for k in ['acc','prec','rec','f1','train_time','pred_time']}
metrics_json['ga_features_selected'] = int(len(ga.selected))
metrics_json['qio_features_selected'] = int(len(qio.selected))
metrics_json['dataset_size'] = len(df)
metrics_json['train_size'] = len(X_train)
metrics_json['test_size'] = len(X_test)
metrics_json['vocab_size'] = len(tfidf.vocabulary_)

with open(f'{FIG_DIR}/metrics.json', 'w') as f:
    json.dump(metrics_json, f, indent=2)

# Print summary table
print("\n" + "=" * 70)
print("CELL 10: Final Summary")
print("=" * 70)
results_df = pd.DataFrame({
    'Model': model_names,
    'Accuracy': [models[m]['acc'] for m in model_names],
    'Precision': [models[m]['prec'] for m in model_names],
    'Recall': [models[m]['rec'] for m in model_names],
    'F1 Score': [models[m]['f1'] for m in model_names],
    'Train(s)': [models[m]['train_time'] for m in model_names],
    'Pred(s)': [models[m]['pred_time'] for m in model_names],
})
print(results_df.to_string(index=False))
print("\nDone! All figures and metrics exported.")


CELL 10: Final Summary
        Model  Accuracy  Precision  Recall  F1 Score   Train(s)  Pred(s)
  Naive Bayes    0.8599   0.847998  0.8770  0.862255  12.845536 0.936841
   Linear SVM    0.8859   0.877667  0.8968  0.887130  13.138045 0.949337
Logistic Reg.    0.8932   0.883834  0.9054  0.894487  12.805335 0.931933
     GA + SVM    0.8267   0.812034  0.8502  0.830679 112.191823 0.007296
    QIO + SVM    0.8218   0.805545  0.8484  0.826417  94.158808 0.007401

Done! All figures and metrics exported.
